# Real Estate Listing-Price Model

## tl;dr

This notebook trains a basic, leakage-safe model on **1,689 listings** from
Argentina and Uruguay. Training-only cross-validation selects **Ridge Regression**.
On the untouched 20% test set, it reaches **R² = 0.475**, **MAE = $105,473**, and
**MAPE = 27.2%**.

This estimates current online listing prices. It does **not** forecast future
market trends because the project does not yet have historical listing snapshots.

## Context and Method

The target is `price_usd`. The model uses property, location, area, room, expense,
and amenity fields. It excludes any feature calculated from the target, such as
`price_per_sqm`.

Basic assumptions:

- Each row is one residential sale listing.
- Amenity value `0` means “not mentioned,” not confirmed absence.
- Uruguay and InfoCasas dominate the sample.
- Preprocessing is fitted only on training data.
- The held-out test set is used once, after model selection.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import KFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
TEST_SIZE = 0.20
CV_FOLDS = 5

working_directory = Path.cwd()
PROJECT_ROOT = (
    working_directory
    if (working_directory / "data").exists()
    else working_directory.parent
)
MODEL_DATA_PATH = PROJECT_ROOT / "data" / "scraped_real_estate_model_features.csv"
FULL_DATA_PATH = PROJECT_ROOT / "data" / "scraped_real_estate_training.csv"

## Data

The compact CSV is the modeling input. The full CSV is loaded only to show source
coverage, because source identifiers are intentionally absent from the compact
feature table.

In [2]:
model_df = pd.read_csv(MODEL_DATA_PATH)
full_df = pd.read_csv(FULL_DATA_PATH)

dataset_summary = pd.DataFrame(
    [
        {
            "dataset": "Full extracted data",
            "rows": len(full_df),
            "columns": len(full_df.columns),
            "purpose": "Scraping evidence and detailed fields",
        },
        {
            "dataset": "Compact model data",
            "rows": len(model_df),
            "columns": len(model_df.columns),
            "purpose": "EDA and price modeling",
        },
    ]
)
display(dataset_summary)

,dataset,rows,columns,purpose
0,Full extracted data,1777,161,Scraping evidence and detailed fields
1,Compact model data,1689,63,EDA and price modeling


In [3]:
quality_checks = pd.DataFrame(
    [
        {"check": "Exact duplicate model rows", "value": int(model_df.duplicated().sum())},
        {"check": "Missing target prices", "value": int(model_df["price_usd"].isna().sum())},
        {"check": "Non-positive target prices", "value": int((model_df["price_usd"] <= 0).sum())},
        {"check": "Argentina model rows", "value": int((model_df["country"] == "AR").sum())},
        {"check": "Uruguay model rows", "value": int((model_df["country"] == "UY").sum())},
    ]
)

source_coverage = (
    full_df["data_source"]
    .fillna("missing")
    .value_counts()
    .rename_axis("source")
    .reset_index(name="rows")
)
source_coverage["share"] = source_coverage["rows"] / source_coverage["rows"].sum()

display(quality_checks)
display(source_coverage.style.format({"share": "{:.1%}"}))

assert quality_checks.loc[quality_checks["check"] == "Exact duplicate model rows", "value"].item() == 0
assert quality_checks.loc[quality_checks["check"] == "Missing target prices", "value"].item() == 0
assert quality_checks.loc[quality_checks["check"] == "Non-positive target prices", "value"].item() == 0

,check,value
0,Exact duplicate model rows,0
1,Missing target prices,0
2,Non-positive target prices,0
3,Argentina model rows,579
4,Uruguay model rows,1110


,source,rows,share
0,infocasas,1185,66.7%
1,zonaprop,581,32.7%
2,argenprop,11,0.6%


The compact data passes the basic checks: no exact duplicate rows and no
missing or non-positive target prices. Coverage is uneven: InfoCasas supplies most
Uruguay rows, ZonaProp supplies most Argentina rows, ArgenProp supplies six rows,
and Gallito supplies none. Treat the result as a baseline for the collected
listings, not the full regional market.

## Results

### 1. Choose Features

Keep only fields available when estimating a new listing price.

In [4]:
TARGET_COLUMN = "price_usd"

NUMERIC_FEATURE_CANDIDATES = [
    "covered_area_sqm", "effective_area_sqm", "total_area_sqm",
    "bedrooms", "bathrooms", "total_rooms", "area_per_room_sqm",
    "amenity_count", "parking_spaces", "expenses_usd", "age_years",
    "floor_number", "floors_in_building", "distance_to_sea_blocks",
    "is_apartment", "is_house", "is_new_construction",
    "is_under_construction", "has_balcony", "has_terrace", "has_garden",
    "has_patio", "has_pool", "has_elevator", "has_security",
    "has_air_conditioning", "has_heating", "has_laundry_room",
    "has_storage_room", "has_gym", "has_grill", "is_furnished",
    "is_gated_community", "is_near_beach", "is_near_park", "is_near_sea",
    "is_near_subway", "pets_allowed", "mortgage_eligible",
]

CATEGORICAL_FEATURE_CANDIDATES = [
    "property_type", "property_subtype", "construction_stage", "country",
    "province", "city", "neighborhood", "location_key", "area_bucket",
    "room_bucket",
]


def prepare_feature_frame(frame):
    '''Create target-independent model features from valid positive-price rows.

    The function keeps only available columns and creates two aggregate counts
    exclusively from listing attributes that exist at prediction time.
    '''

    valid = frame.loc[
        frame[TARGET_COLUMN].notna() & frame[TARGET_COLUMN].gt(0)
    ].copy()
    numeric_features = [
        column
        for column in NUMERIC_FEATURE_CANDIDATES
        if column in valid.columns and valid[column].notna().any()
    ]
    categorical_features = [
        column
        for column in CATEGORICAL_FEATURE_CANDIDATES
        if column in valid.columns and valid[column].notna().any()
    ]

    features = valid[numeric_features + categorical_features].copy()
    luxury_columns = [
        column
        for column in ("has_pool", "has_gym", "has_grill", "has_security")
        if column in features.columns
    ]
    proximity_columns = [
        column
        for column in ("is_near_beach", "is_near_sea", "is_near_park", "is_near_subway")
        if column in features.columns
    ]

    features["luxury_amenity_count"] = features[luxury_columns].fillna(0).sum(axis=1)
    features["location_proximity_count"] = features[proximity_columns].fillna(0).sum(axis=1)
    numeric_features += ["luxury_amenity_count", "location_proximity_count"]

    target = valid[TARGET_COLUMN].astype(float)
    return features, target, numeric_features, categorical_features


features, target, numeric_features, categorical_features = prepare_feature_frame(model_df)

assert TARGET_COLUMN not in features.columns
assert "price_per_sqm" not in features.columns

pd.DataFrame(
    {
        "usable_rows": [len(features)],
        "numeric_features": [len(numeric_features)],
        "categorical_features": [len(categorical_features)],
    }
)

,usable_rows,numeric_features,categorical_features
0,1689,38,10


### 2. Split and Prepare the Data

Keep 20% of the rows untouched for the final check.

In [5]:
x_train, x_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)


def build_preprocessor(numeric_columns, categorical_columns):
    '''Build preprocessing that is fitted only inside each training fold.

    Numeric values receive median imputation and scaling. Categorical values
    receive most-frequent imputation and unknown-safe one-hot encoding.
    '''

    numeric_pipeline = Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )
    categorical_pipeline = Pipeline(
        [
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]
    )
    return ColumnTransformer(
        [
            ("numeric", numeric_pipeline, numeric_columns),
            ("categorical", categorical_pipeline, categorical_columns),
        ],
        verbose_feature_names_out=False,
    )


def build_estimator(model):
    '''Combine train-fitted preprocessing with a log-target regressor.

    The log transform limits domination by a few very expensive listings while
    predictions and reported errors remain in original US dollars.
    '''

    pipeline = Pipeline(
        [
            ("preprocessor", build_preprocessor(numeric_features, categorical_features)),
            ("model", clone(model)),
        ]
    )
    return TransformedTargetRegressor(
        regressor=pipeline,
        func=np.log1p,
        inverse_func=np.expm1,
        check_inverse=False,
    )


candidates = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=10.0),
    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        max_depth=20,
        min_samples_split=5,
        random_state=RANDOM_STATE,
        n_jobs=1,
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=150,
        max_depth=4,
        learning_rate=0.05,
        random_state=RANDOM_STATE,
    ),
}

pd.DataFrame({"split": ["Train", "Test"], "rows": [len(x_train), len(x_test)]})

,split,rows
0,Train,1351
1,Test,338


### 3. Compare Basic Models

Use five-fold cross-validation on the training rows only.

In [6]:
splitter = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
cv_rows = []
estimators = {}

for model_name, model in candidates.items():
    estimator = build_estimator(model)
    scores = cross_validate(
        estimator,
        x_train,
        y_train,
        cv=splitter,
        scoring={"r2": "r2", "mae": "neg_mean_absolute_error"},
        n_jobs=1,
        error_score="raise",
    )
    cv_rows.append(
        {
            "model": model_name,
            "CV mean R2": scores["test_r2"].mean(),
            "CV R2 std": scores["test_r2"].std(),
            "CV mean MAE (USD)": -scores["test_mae"].mean(),
        }
    )
    estimators[model_name] = estimator

cv_results = (
    pd.DataFrame(cv_rows)
    .sort_values("CV mean R2", ascending=False)
    .reset_index(drop=True)
)
best_model_name = cv_results.loc[0, "model"]
display(
    cv_results.style.format(
        {
            "CV mean R2": "{:.3f}",
            "CV R2 std": "{:.3f}",
            "CV mean MAE (USD)": "${:,.0f}",
        }
    )
)

,model,CV mean R2,CV R2 std,CV mean MAE (USD)
0,Ridge Regression,0.628,0.159,"$107,995"
1,Gradient Boosting,0.428,0.192,"$116,350"
2,Random Forest,0.406,0.206,"$114,262"
3,Linear Regression,0.185,0.547,"$126,356"


### 4. Check the Selected Model

Fit the selected model once and evaluate it on the held-out rows.

In [7]:
best_estimator = estimators[best_model_name]
best_estimator.fit(x_train, y_train)
test_predictions = np.maximum(best_estimator.predict(x_test), 0)

test_metrics = {
    "R2": r2_score(y_test, test_predictions),
    "MAE (USD)": mean_absolute_error(y_test, test_predictions),
    "RMSE (USD)": np.sqrt(mean_squared_error(y_test, test_predictions)),
    "MAPE": mean_absolute_percentage_error(y_test, test_predictions),
}
metrics_table = pd.DataFrame(
    {
        "metric": list(test_metrics),
        "value": list(test_metrics.values()),
    }
)

print(f"Selected model: {best_model_name}")
display(metrics_table.style.format({"value": "{:,.3f}"}))

test_segments = model_df.loc[x_test.index, ["country", "property_type"]].reset_index(drop=True)
actual_test_prices = y_test.to_numpy()
segment_rows = []
segment_definitions = {
    "Argentina": test_segments["country"].eq("AR"),
    "Uruguay": test_segments["country"].eq("UY"),
    "Apartment": test_segments["property_type"].eq("Apartment"),
    "House": test_segments["property_type"].eq("House"),
}
for segment_name, segment_mask in segment_definitions.items():
    mask = segment_mask.to_numpy()
    if mask.sum() < 2 or np.unique(actual_test_prices[mask]).size < 2:
        continue
    segment_rows.append(
        {
            "segment": segment_name,
            "rows": int(mask.sum()),
            "R2": r2_score(actual_test_prices[mask], test_predictions[mask]),
            "MAE (USD)": mean_absolute_error(actual_test_prices[mask], test_predictions[mask]),
            "MAPE": mean_absolute_percentage_error(actual_test_prices[mask], test_predictions[mask]),
        }
    )
segment_metrics = pd.DataFrame(segment_rows)
display(
    segment_metrics.style.format(
        {"R2": "{:.3f}", "MAE (USD)": "${:,.0f}", "MAPE": "{:.1%}"}
    )
)

assert best_model_name == "Ridge Regression"
assert 0 <= test_metrics["R2"] <= 1
assert test_metrics["MAE (USD)"] > 0
assert test_metrics["MAPE"] > 0

Selected model: Ridge Regression


,metric,value
0,R2,0.475
1,MAE (USD),"105,473.124"
2,RMSE (USD),"307,036.826"
3,MAPE,0.272


,segment,rows,R2,MAE (USD),MAPE
0,Argentina,110,0.684,"$84,985",31.8%
1,Uruguay,228,0.442,"$115,358",25.0%
2,Apartment,258,0.522,"$74,156",21.0%
3,House,80,0.168,"$206,470",47.0%


### 5. Review the Results

The charts compare models, actual and predicted prices, residuals, and the largest Ridge weights.

In [8]:
fitted_pipeline = best_estimator.regressor_
preprocessor = fitted_pipeline.named_steps["preprocessor"]
fitted_model = fitted_pipeline.named_steps["model"]
feature_names = preprocessor.get_feature_names_out()
feature_weights = np.abs(np.asarray(fitted_model.coef_).reshape(-1))
importance = (
    pd.DataFrame({"feature": feature_names, "weight": feature_weights})
    .sort_values("weight", ascending=False)
    .head(12)
)

figure, axes = plt.subplots(2, 2, figsize=(14, 10))

ordered = cv_results.sort_values("CV mean R2")
colors = ["#d4a72c" if name == best_model_name else "#3b82b8" for name in ordered["model"]]
axes[0, 0].barh(
    ordered["model"],
    ordered["CV mean R2"],
    xerr=ordered["CV R2 std"],
    color=colors,
)
axes[0, 0].set_title("Training-only cross-validation R²")
axes[0, 0].set_xlabel("Mean R² (error bars: 1 standard deviation)")
axes[0, 0].grid(axis="x", alpha=0.25)

axes[0, 1].scatter(y_test, test_predictions, alpha=0.55, s=24)
plot_min = float(min(y_test.min(), test_predictions.min()))
plot_max = float(max(y_test.max(), test_predictions.max()))
axes[0, 1].plot([plot_min, plot_max], [plot_min, plot_max], "--", color="#c43d3d")
axes[0, 1].set_xscale("log")
axes[0, 1].set_yscale("log")
axes[0, 1].set_title(f"Held-out test: {best_model_name}")
axes[0, 1].set_xlabel("Actual price (USD, log scale)")
axes[0, 1].set_ylabel("Predicted price (USD, log scale)")
axes[0, 1].grid(alpha=0.25)

residuals = y_test.to_numpy() - test_predictions
axes[1, 0].hist(residuals, bins=35, color="#7b5ea7", edgecolor="white")
axes[1, 0].axvline(0, linestyle="--", color="#c43d3d")
axes[1, 0].set_title("Held-out residuals")
axes[1, 0].set_xlabel("Actual - predicted price (USD)")
axes[1, 0].set_ylabel("Listings")
axes[1, 0].grid(axis="y", alpha=0.25)

top_weights = importance.sort_values("weight")
axes[1, 1].barh(top_weights["feature"], top_weights["weight"], color="#16826c")
axes[1, 1].set_title("Largest absolute Ridge coefficients")
axes[1, 1].set_xlabel("Absolute coefficient on log-price target")
axes[1, 1].grid(axis="x", alpha=0.25)

figure.suptitle(
    "Leakage-safe listing-price baseline\n"
    f"Test R²={test_metrics['R2']:.3f} | "
    f"MAE=${test_metrics['MAE (USD)']:,.0f} | "
    f"MAPE={test_metrics['MAPE']:.1%}",
    fontsize=15,
    fontweight="bold",
)
figure.tight_layout(rect=(0, 0, 1, 0.94))
plt.show()

C:\Users\Daniel\AppData\Local\Temp\ipykernel_41144\3865983074.py:60: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Takeaways

- Ridge Regression is the strongest tested baseline, with mean cross-validation
  R² of about **0.628**.
- The held-out result is **R² = 0.475**, **MAE = $105,473**, and **MAPE = 27.2%**.
- These values are regression errors, not “accuracy.”
- The added Argentina house listings improve coverage but make the broader test set harder;
  use the segment table to avoid hiding weak groups behind one overall score.
- Location, size, bathrooms, and room/area groups carry useful signal.
- Results are limited by source, country, and property-type imbalance.
- True future forecasting requires dated listing snapshots aligned with economic
  indicators over time.